# Benchmark (2): can an LLM correctly explain why the portfolio beat the S&P 500?

 Give an LLM the LLM-S ACE screening rules, the screened stocks, the residual nodewise / GMV
weights, the return sequences and the fundamentals data (`firm_data.db`, with one-month forward returns). Ask it to
explain in detail why the portfolio (Sharpe 0.8931) outperformed the S&P 500 (0.2685). Then check by hand whether each
part of the explanation is right, and score it 0–2. One run.

**Design**
| Step | What | Code |
|---|---|---|
| 1 | Assemble the preprint's LLM-S ACE output files into one DuckDB: holdings with tickers, industries, characteristics and next-month returns; returns; screens; strategies; `firms`. | `benchmarks/b2_attribution.py::load` |
| 2 | **Explainer** (Claude Opus 5.5) with a read-only SQL tool. It must return a narrative **plus typed claims** (portfolio, metric, entity, period, value), so each claim can be recomputed. | `benchmarks/b2_explain.py` |
| 3 | **Automatic verification**: every typed claim is recomputed from the data → match / sign-only / mismatch / not auto-checkable. | `b2_attribution.py::verify_claims` |
| 4 | **Fact sheet** (the deterministic attribution: Sharpe decomposition, by year, leave-one-quarter-out, CAPM, stock/leg/industry contributions, exposures), for checking the claims that cannot be verified automatically. | `b2_attribution.py::fact_sheet` |
| 5 | **Human score** 0–2 per claim and overall. | this notebook |

**How to run:** kernel *agentic-screen (Python 3.11)*, then *Run All* (about 30 s; it loads the saved explanation).
Set `RUN_EXPLAINER = True` to ask the LLM again (a few minutes, ~$1–3).

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
HERE = Path.cwd().resolve()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "benchmarks" / "common.py").exists())   # repo root
sys.path.insert(0, str(REPO))
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 140, "display.precision", 4)

from benchmarks.b2_attribution import load, fact_sheet, verify_claims, explainer_prompt, TABLE_DOCS
from benchmarks.b2_explain import explain, load_explanation, review_table, B2
from benchmarks.common import cost_usd, RESULTS
RUN_EXPLAINER = False
D = load(); F = fact_sheet(D)
print({k: len(v) for k, v in D.items() if hasattr(v, "__len__") and k != "con"})

## 1. Reproduce the reported numbers first

Weights at date T × each stock's return over month T+1 reproduce the reported return series. The gap to the CSV is the
10 bp × turnover cost. One inconsistency to note: **0.8931 uses the sample standard deviation (ddof=1),
while 0.2685 for the S&P uses the population one (ddof=0)**. On the same convention the S&P Sharpe ratio is 0.2642.

In [ ]:
display(F["headline"])
gross = D["ace"].groupby("return_month").contribution.sum().values
print("corr(sum w*r, reported net returns) =", np.corrcoef(gross, D["rets"].ace)[0, 1].round(5),
      "| mean monthly cost drag =", (gross - D["rets"].ace).mean().round(5))

## 2. The LLM's explanation

In [ ]:
if RUN_EXPLAINER:
    R = explain(D)
R = load_explanation()    # the saved run; falls back to the committed copy in benchmarks/results/
cost = f"~${cost_usd(R['usage']):.2f}" if "usage" in R else "cost in the full transcript"
print(f"model {R['model']} | {R.get('sql_queries', '?')} SQL queries | {len(R['claims'])} claims | {cost}")
display(Markdown("### Summary\n" + R["summary"]))
display(Markdown("### Explanation\n" + R["explanation"]))
display(Markdown("### Caveats (its own)\n" + R["caveats"]))

## 3. Automatic verification of each claim

`recomputed` is the value from our own code. *match* = within tolerance (10% relative or a metric-specific absolute
tolerance); *sign/rank only* = right direction, wrong size; *mismatch* = wrong; *not auto-checkable* = qualitative, or
not expressible as a typed metric. Those need the fact sheet below and a human.

In [ ]:
T = review_table(R, D)
display(T[["id", "kind", "claim", "metric", "entity", "period", "claimed", "recomputed", "auto_check"]])
display(T.auto_check.value_counts().to_frame("claims"))

## 4. Fact sheet, for checking the rest by hand

The deterministic attribution, computed independently of the LLM.

In [ ]:
for k in ["sharpe_gap", "capm_ace_on_sp500", "by_year", "leave_one_quarter_out"]:
    display(Markdown(f"**{k}**")); display(F[k])

In [ ]:
for k in ["top_stocks", "bottom_stocks", "top_stock_each_year", "legs", "legs_by_sign"]:
    display(Markdown(f"**{k}**")); display(F[k])

In [ ]:
for k in ["industry", "characteristic_exposure", "portfolio_shape_by_year", "best_worst_months"]:
    display(Markdown(f"**{k}**")); display(F[k])

**Questions to check the explanation against** (from the fact sheet):
1. Does it say the Sharpe gap comes almost entirely from a **higher mean**, not lower volatility?
2. Does it find that the outperformance is concentrated in **2022** (ACE +27.7% vs S&P −18.2%), and that ACE **lagged in 2023**?
3. Does it find the **concentration**: XOM ≈ 57% and LLY ≈ 34% of gross contribution; Sharpe without XOM ≈ 0.50?
4. Does it notice that the profit came from names the screen labelled **SELL but the optimizer held long**
   (the "direction-agnostic" design: labels do not fix the sign)?
5. Does it attribute it to an **energy overweight** (allocation) rather than stock selection within industries?
6. Does it flag the **Sharpe convention inconsistency** and the low statistical strength (CAPM alpha t ≈ 1.5, 31 months)?

## 5. Human scoring (0–2)

Score each claim (0 = wrong or unsupported, 1 = partly right, 2 = right and supported), then the explanation as a
whole. Saved to `benchmarks/results/`.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
T.to_csv(RESULTS / "b2_claims_review_sheet.csv", index=False)
auto = T[T.auto_check != "not auto-checkable"]
print(f"auto-checkable claims: {len(auto)}/{len(T)}; matched: {(auto.auto_check == 'match').sum()}; "
      f"mismatched: {auto.auto_check.str.startswith('mismatch').sum()}")
(RESULTS / "b2_explanation.json").write_text(json.dumps({k: R[k] for k in ("model", "summary", "explanation", "caveats", "claims")}, indent=2))
print("saved:", RESULTS / "b2_claims_review_sheet.csv")